# Part 6 · os / pathlib / shutil / glob / subprocess
> 文件系统操作、路径处理、环境变量、子进程

## 1. pathlib.Path（推荐，现代写法）

In [ ]:
from pathlib import Path

# --- 创建路径对象 ---
p = Path('data/orders.csv')          # 相对路径
p = Path('/home/user/data')          # 绝对路径
p = Path.home()                      # 用户主目录 /home/user
p = Path.cwd()                       # 当前工作目录
p = Path(__file__).parent            # 脚本所在目录

# --- 路径拼接 ---
p = Path('/data') / 'orders' / 'jan.csv'  # 用 / 拼接（推荐）
p.joinpath('subdir', 'file.txt')          # 等价方法

# --- 路径属性 ---
p.name        # 'jan.csv'          文件名（含扩展名）
p.stem        # 'jan'              文件名（不含扩展名）
p.suffix      # '.csv'             扩展名
p.suffixes    # ['.tar', '.gz']    多重扩展名
p.parent      # Path('/data/orders')  父目录
p.parents[0]  # 父目录
p.parents[1]  # 祖父目录
p.parts       # ('/', 'data', 'orders', 'jan.csv')
p.anchor      # '/'                根目录

# --- 判断 ---
p.exists()        # 路径是否存在
p.is_file()       # 是否为文件
p.is_dir()        # 是否为目录
p.is_absolute()   # 是否为绝对路径
p.is_symlink()    # 是否为符号链接

# --- 读写（小文件）---
text = p.read_text(encoding='utf-8')        # 读全部文本
p.write_text('hello\n', encoding='utf-8')   # 写文本（覆盖）
data = p.read_bytes()                       # 读全部字节
p.write_bytes(b'data')                      # 写字节

# --- 目录操作 ---
p.mkdir()                         # 创建目录（已存在则报错）
p.mkdir(parents=True, exist_ok=True)  # 递归创建，存在不报错
p.rmdir()                         # 删除空目录

# --- 文件操作 ---
p.unlink()                        # 删除文件（不存在报错）
p.unlink(missing_ok=True)         # 删除文件（不存在不报错，3.8+）
p.rename(Path('new_name.csv'))    # 移动/重命名
p.replace(Path('new_name.csv'))   # 移动（目标存在则覆盖）
p.touch()                         # 创建空文件，或更新修改时间

# --- 遍历 ---
list(Path('/data').iterdir())               # 列出目录下所有（文件+目录）
list(Path('/data').glob('*.csv'))           # 匹配 CSV 文件
list(Path('/data').glob('**/*.csv'))        # 递归匹配（等价 rglob）
list(Path('/data').rglob('*.parquet'))      # 递归搜索
[f for f in Path('/data').iterdir() if f.is_file()]  # 只要文件

# --- 文件信息 ---
stat = p.stat()
stat.st_size          # 文件大小（字节）
stat.st_mtime         # 修改时间（Unix 时间戳）
p.stat().st_size / 1024**2  # MB

# --- 其他 ---
p.resolve()                    # 转为绝对路径（解析符号链接）
p.relative_to('/home/user')    # 转为相对路径
p.with_name('new.csv')         # 替换文件名
p.with_stem('new')             # 替换文件名（不含扩展名）
p.with_suffix('.parquet')      # 替换扩展名
str(p)                         # 转为字符串（传给不支持 Path 的库）

## 2. os — 底层操作

In [ ]:
import os

# --- 目录操作 ---
os.getcwd()                          # 当前目录
os.chdir('/tmp')                     # 切换目录
os.listdir('.')                      # 列出当前目录（返回名称列表）
os.mkdir('newdir')                   # 创建目录
os.makedirs('a/b/c', exist_ok=True) # 递归创建
os.rmdir('emptydir')                 # 删除空目录

# --- 文件操作 ---
os.remove('file.txt')                # 删除文件
os.rename('old.txt', 'new.txt')      # 重命名/移动
os.replace('src', 'dst')             # 移动（目标存在则覆盖）

# --- 路径操作（推荐用 pathlib 代替）---
os.path.join('/data', 'orders', 'jan.csv')  # 路径拼接
os.path.exists('/data/file.csv')     # 是否存在
os.path.isfile('/data/file.csv')     # 是否为文件
os.path.isdir('/data')               # 是否为目录
os.path.dirname('/data/file.csv')    # '/data'
os.path.basename('/data/file.csv')   # 'file.csv'
os.path.splitext('file.csv')         # ('file', '.csv')
os.path.abspath('./file.csv')        # 绝对路径
os.path.expanduser('~/data')         # 展开 ~
os.path.getsize('file.csv')          # 文件大小（字节）
os.path.getmtime('file.csv')         # 修改时间（时间戳）

# --- os.walk（递归遍历）---
for root, dirs, files in os.walk('/data'):
    for f in files:
        full_path = os.path.join(root, f)
        print(full_path)
    dirs[:] = [d for d in dirs if not d.startswith('.')]  # 跳过隐藏目录

# --- 环境变量 ---
os.environ                           # 所有环境变量（类字典）
os.environ['HOME']                   # 读（不存在抛 KeyError）
os.environ.get('DB_URL', 'default')  # 有默认值（推荐）
os.environ['MY_VAR'] = 'value'       # 设置（仅当前进程）
os.getenv('DB_URL', 'default')       # 等价 environ.get
os.unsetenv('MY_VAR')                # 删除

# --- 进程信息 ---
os.getpid()                          # 当前进程 PID
os.getppid()                         # 父进程 PID
os.cpu_count()                       # CPU 核数

## 3. shutil — 高级文件操作

In [ ]:
import shutil

# --- 复制 ---
shutil.copy('src.txt', 'dst.txt')       # 复制文件内容 + 权限
shutil.copy2('src.txt', 'dst.txt')      # 同上 + 保留元数据（时间戳等）
shutil.copyfile('src.txt', 'dst.txt')   # 只复制内容
shutil.copytree('src_dir', 'dst_dir')   # 递归复制目录
shutil.copytree('src', 'dst', dirs_exist_ok=True)  # 目标存在时合并（3.8+）

# --- 移动 ---
shutil.move('src.txt', 'dst_dir/')      # 移动文件或目录

# --- 删除 ---
shutil.rmtree('dir_to_delete')          # 递归删除目录及内容（⚠️ 不可恢复）
shutil.rmtree('dir', ignore_errors=True)  # 忽略错误

# --- 压缩/解压 ---
shutil.make_archive('backup', 'zip', 'src_dir')    # 压缩目录
shutil.make_archive('backup', 'gztar', 'src_dir')  # .tar.gz
shutil.unpack_archive('backup.zip', 'dst_dir')     # 解压

# --- 磁盘空间 ---
total, used, free = shutil.disk_usage('/')
print(f'Free: {free / 1024**3:.1f} GB')

# --- 查找可执行文件 ---
shutil.which('python3')   # '/usr/bin/python3'  等价 which 命令

## 4. glob & fnmatch — 文件匹配

In [ ]:
import glob
import fnmatch

# glob.glob
glob.glob('/data/*.csv')                 # 所有 csv
glob.glob('/data/**/*.parquet', recursive=True)  # 递归搜索
glob.glob('/data/orders_202[0-9]*.csv') # 通配符范围

# glob.iglob（惰性，省内存）
for f in glob.iglob('/data/**/*.csv', recursive=True):
    process(f)

# fnmatch（字符串模式匹配）
fnmatch.fnmatch('orders_2024.csv', 'orders_*.csv')   # True
fnmatch.filter(['a.csv','b.txt','c.csv'], '*.csv')   # ['a.csv','c.csv']

## 5. subprocess — 执行系统命令

In [ ]:
import subprocess

# --- subprocess.run（推荐，阻塞等待完成）---
result = subprocess.run(['ls', '-la'], capture_output=True, text=True)
result.returncode   # 0 = 成功
result.stdout       # 标准输出（字符串）
result.stderr       # 错误输出

# check=True：非0退出码自动抛 CalledProcessError
subprocess.run(['python', 'script.py'], check=True)

# 传入字符串（用 shell=True，⚠️ 注意注入风险）
subprocess.run('ls -la | grep csv', shell=True)

# 传入环境变量
subprocess.run(['script.sh'], env={**os.environ, 'MY_VAR': 'value'})

# 工作目录
subprocess.run(['make'], cwd='/path/to/project')

# 获取输出（简便方式）
output = subprocess.check_output(['date'], text=True).strip()

# --- 实时输出（流式读取）---
proc = subprocess.Popen(
    ['tail', '-f', 'app.log'],
    stdout=subprocess.PIPE,
    text=True
)
for line in proc.stdout:
    print(line, end='')
proc.terminate()

## 6. tempfile — 临时文件

In [ ]:
import tempfile

# 临时文件（退出 with 块自动删除）
with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=True) as f:
    f.write('data')
    print(f.name)    # /tmp/tmpXXXXXX.csv

# 临时目录
with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / 'output.parquet'
    df.to_parquet(path)
    # 退出后自动删除整个目录

# 获取系统临时目录
tempfile.gettempdir()   # '/tmp'